**epact_wheel**
======

<img src="large_epact_wheel.jpg" width=500px alt="Gregorian lunar calendar">

A model of the Gregorian lunar calendar
---------------------------------------

This python program calculates two parts of the Gregorian lunar calendar:

(1) A calendar wheel

(2) A lunar wheel indicating new or full moons

Both share the same axis.
The lunar wheel can be printed separately on a transparency to enable tuning the lunar calendar for a given year.

Usage
-----

First mount the lunar wheel on the calendar wheel, so that they both have the same axis.

Then rotate the lunar wheel until the 'hundreds' for a given year are listed between the two gauge lines (e.g. '20' for the year 2026,
or 120 for the year 12012). For reasons of readability the numbers are set at two different radii. A full circle corresponds to 
about 90000 years.

To find out the days of the new or full moon, one needs to calculate the **golden number**

$GN = year ~ modulo ~ 19 + 1$

(e.g. 2026 correponds to GN=13 and so on). The 19 concentric rows of the calendar disk correspond the golden number, counting from the inside to the outside. New or Full moons in the calendar gap actually occur in the next outward row (or the first row, in case GN=19).

Switches:

    separate  *False*: both a calendar for a given century is plotted, with both elements together
              *True*:  two separate plots are created as described above

    full_moon *False*: a new moon disk is created (Luna I, open circles)
              *True*: a full moon disk is created (Luna XIV, full circles)
              
The assignment of moons to days is controlled by the variable *depak* (Delta epact). This is only relevant when separate=False (for the combined output).




Explanation
-----------

In the Gregorian calendar, every day of the 365-day year is assigned a quantity called 'epact', which is actually the age (or phase, numbered Luna I to Luna XXX) of the moon on January 1st. New moons occur according to the Gregorian rules on days which have been assigned the epact of the year.

Example: the epact of the year 2026 is XI. In [this table](https://echo.mpiwg-berlin.mpg.de/ECHOdocuView?url=/mpiwg/online/permanent/library/YXK9FE9W/pageimg&start=71&viewMode=auto&mode=imagepath&pn=80) (scan available at the Max Planck Institute for the History of Science) the calendar commission has published the assignments for January. It can be seen from the column 'cyclus epactarum'  that the first new moon of the year occurs on January 20. For more information on epacts and the Gregorian calendar, see, e.g., [Bien (2007)](https://ui.adsabs.harvard.edu/abs/2007AHES...61...39B/abstract).

The distribution of lunar phases across the year was done by distributing sets of 30 units continuously across the year. The units are often called [**tithis**](https://en.wikipedia.org/wiki/Tithi), sometimes also  **lunar days** or **epactal days**. In the
Meton cycle there are 235 lunar months in 19 years: 19 x 365.25 / 235 = 29.530851 days
(the modern value is 29.530589 days). A lunar unit thus is 29.530851 days / 30 = 0,9843617 days.

The sets of the three lunar units from epact XXIIII to XXVI are assigned to only two days at six particular times in the year according to Gregorian rules. Due to these six "compressions" we have six 'full' lunar months of 30 days and six 'hollow' lunar months of 29 days,
leading to 6 x 30 + 6 x 29=354 days (starting, e.g., with a new moon on January 1st).
This cycle of moon phases is 11 days shorter than a Gregorian year, so that the epact (moon phase on January 1st) of the next year is increased by 11 each year (but 12 in the transition from GN=19 to GN=1, this is called jump of the moon).
Over the cycle of 19 years in fact there are an additional of 7 intercalary lunar
months inserted for the total of 235 lunar months.
Note also that in leap years February 24 is two days long (it is called the bissextus).

**epact_wheel** creates a model of the Gregorian lunar cycle with the following properties:

1. the calendar wheel features 13 x 30 = 390 tithis (one full circle), of which 365 are assigned to the days of the year.
   Moons in the gap appear again in January for the next golden number (after GN 19 go back to GN 1).

   Note especially the six sets of the Gregorian exception rules where 3 lunar units are assigned to two days. The unit
   in the middle is assigned differently, depending on the golden number (indicated by the thick line).
       
2. on the calendar wheel there are 19 circles (numbered from the inside out) that correspond to the golden numbers

3. the lunar wheel features 13 new or full moons, each spaced by 30 lunar units, for each of the 19 golden numbers

4. the new or full moons for a given year can be determined by rotating the lunar wheel until the
   'hundreds' for a given year are listed between the two gauge lines (e.g. '20' for the year 2023,
    or 120 for the year 12012). 

    The logic

    counter-clockwise three times in 400 years (1700, 1800, 1900, not 2000, ...)
    
    clockwise eight times in 2500 years (1800,2100,....,3900,4300, ...)
    

A special phase assignment is done at the end of the 19th year (outermost circle): A full moon at the 366th position is counted as December 31st (because otherwise one full moon would be missing in the changeover from golden number 19 to 1).

<img src="large_epact_wheel_zoom.jpg" width=300px alt="Gregorian lunar calendar">

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# function definitions

def golden_number(year):
    return year%19+1

def create_epacts(century):
    year=century - 1
    principal_epact = calculate_principal_epact(century)
    
    epacts = np.arange(1,1+19*11,11)
    epacts =  (epacts + principal_epact)
    #epacts[epacts==0]=30
    return epacts

def calculate_principal_epact(century):
    sun=np.floor(3*century/4)-12                     # solar equation
    moon=np.floor((8*century+5)/25)-5                # lunar equation
    return int(-sun + moon)

# number of tithis to rotate relative to the year 2000
def calculate_delta(century):
    sun=np.floor(3*century/4)-12                     # solar equation
    moon=np.floor((8*century+5)/25)-5                # lunar equation
    return -sun + moon + 2

# epact wheel
def create_wheel(ax):
    
    phis=np.deg2rad(dphi*np.arange(372))
       
    for gn in range(20):
            
        rtop=r1-gn*delta
        ax.plot(x0+rtop*np.cos(phis),y0+rtop*np.sin(phis),color='k',linewidth=0.5)
        
    for phi in phis:
        rtop=r1
        rbot=r1-19*delta
        ax.plot([x0+rtop*np.cos(phi),x0+rbot*np.cos(phi)],
                [y0+rtop*np.sin(phi),y0+rbot*np.sin(phi)],color='k',linewidth=0.5)
   
    # limits of solar months, Gregorian exceptions depend on full or new moon
    if full_moon:
        phis=np.deg2rad(dphi*np.array([0, 31, 60, 91, 122, 153, 184, 215, 247, 277, 309, 339, 371]))
    else: 
        phis=np.deg2rad(dphi*np.array([0, 31, 60, 91, 122, 153, 184, 216, 247, 278, 309, 340, 371]))

    rtop=r1
    rbot=r1-19*delta    
    for phi in phis:
        ax.plot([x0+rtop*np.cos(phi),x0+rbot*np.cos(phi)],
                [y0+rtop*np.sin(phi),y0+rbot*np.sin(phi)],color='red',linewidth=2.0)
    
    labels=['1.1.','1.2.','1.3.','1.4.','1.5.','1.6.','1.7.','1.8.','1.9.','1.10.','1.11.','1.12.']
    rotation=np.rad2deg(phis)
    rtop=r1+3*delta
    for x in range(12):
        ax.text(x0+rtop*np.cos(phis[x]),y0+rtop*np.sin(phis[x]),labels[x],rotation=rotation[x],fontsize=16,
               horizontalalignment='center',verticalalignment='center')

    # add day numbers
    rbot=r1-19.7*delta
    for x in range(11):
        if x in (1,3,5):             # one more tithi if exception present
            dayoff=np.deg2rad(dphi)
        else:
            dayoff=0
        
        delphi=np.deg2rad(dphi*9.4)
        ax.text(x0+rbot*np.cos(phis[x]+delphi+dayoff),y0+rbot*np.sin(phis[x]+delphi+dayoff),'10',rotation=rotation[x]+9,
            fontsize=5,horizontalalignment='center',verticalalignment='center',color='red')
        
        delphi=np.deg2rad(dphi*19.4)
        ax.text(x0+rbot*np.cos(phis[x]+delphi+dayoff),y0+rbot*np.sin(phis[x]+delphi+dayoff),'20',rotation=rotation[x]+19,
            fontsize=5,horizontalalignment='center',verticalalignment='center',color='red')
        
    # Gregorian exceptions for hollow months (every 60 tithis)
    for phase in range(48,48+6*60,60):
        if not full_moon:
            phase=phase-13
        phis=np.deg2rad(np.array([dphi*(phase-1),dphi*(phase+2)]))
        rtop=r1
        rbot=r1-19*delta
        for phi in phis:
            ax.plot([x0+rtop*np.cos(phi),x0+rbot*np.cos(phi)],
                [y0+rtop*np.sin(phi),y0+rbot*np.sin(phi)],color='k',linewidth=1.5)

        phi=np.deg2rad(dphi*phase)
        rtop=r1-8*delta
        ax.plot([x0+rtop*np.cos(phi),x0+rbot*np.cos(phi)],
            [y0+rtop*np.sin(phi),y0+rbot*np.sin(phi)],color='k',linewidth=1.5)
        phi2=np.deg2rad(dphi*(phase+1))
        ax.plot([x0+rtop*np.cos(phi),x0+rtop*np.cos(phi2)],
            [y0+rtop*np.sin(phi),y0+rtop*np.sin(phi2)],color='k',linewidth=1.5)
        
        rbot=rtop
        rtop=r1
        ax.plot([x0+rbot*np.cos(phi2),x0+rtop*np.cos(phi2)],
            [y0+rbot*np.sin(phi2),y0+rtop*np.sin(phi2)],color='k',linewidth=1.5)
    
    # epact 19 exception
    phis=np.deg2rad([dphi*(371-1),dphi*372])
    for gn in range(2):
        rtop=r1-gn*delta
        ax.plot(x0+rtop*np.cos(phis),y0+rtop*np.sin(phis),color='k',linewidth=1.5)
    for phi in phis:
        rtop=r1
        rbot=r1-delta
        ax.plot([x0+rtop*np.cos(phi),x0+rbot*np.cos(phi)],
                [y0+rtop*np.sin(phi),y0+rbot*np.sin(phi)],color='k',linewidth=1.5)

    # line to read off year
    ax.plot([x0+r1*6/18,x0+r1*10/18],[y0+24,y0+24],color='k',linewidth=0.2)
    ax.plot([x0+r1*6/18,x0+r1*10/18],[y0-14,y0-14],color='k',linewidth=0.2)
    
    _=ax.axis('off')
    #ax.set_title('luna xiv (full mooon)', fontsize=16)

def rad(gn):
    return r1+(gn-19)*delta
    
# moveable lunar disc
def create_moons(ax2, depak):
    offset=14+depak           # made to fit epact 0 on Jan 1st with 14+depak
    for gn in range(1,20):
        
        rtop=rad(gn)-0.5*delta
        if full_moon:
            phis=np.deg2rad(dphi*(np.arange(0,390,30)+offset)+0.5*dphi)
            ax2.scatter(x0+rtop*np.cos(phis),y0+rtop*np.sin(phis),marker='o',color='blue',s=25.0)
        else:       
            phis=np.deg2rad(dphi*(np.arange(0,390,30)+offset-13)+0.5*dphi)
            ax2.scatter(x0+rtop*np.cos(phis),y0+rtop*np.sin(phis),facecolors='none',edgecolors='blue',s=25)
        
        offset=offset-11
    
    # labels for years
    xlabel=x0+r1/3
    ylabel=y0
    # determine labels for years at the start of centuries
    labels=390*['']
    for century in range(17,922): # 922 1066):
        principal_epact = (-calculate_principal_epact(century) - 2 - depak)%390
        if labels[principal_epact]!='':
            labels[principal_epact]+=','
        labels[principal_epact]+=str(century - 1) 
    # plot labels
    for angle in range(0,len(labels)):
        rotation=angle*dphi
        
        roff=0
        if angle%2==0:
            roff=-5*delta
        
        ax2.text(x0+(r1/2+roff)*np.cos(np.deg2rad(rotation)),y0-(r1/2+roff)*np.sin(np.deg2rad(rotation)),
                 labels[int(angle)],fontsize = 5,rotation=360-rotation,
                 horizontalalignment='center',verticalalignment='center')
    
    
    # label epact offset
    ax2.text(0.1*x0,0.2*y0,'$\Delta$ ='+str(depak),fontsize=20)

In [ ]:
### run this cell to create the plot(s)
# use 'separate' to either create one or two plots (two is for producing a physical copy)

# create separate plots by using separate = True
separate=False
full_moon=True

# lunar age difference for the first of January with respect to the year 2000
depak= 0 #61

# plot variables
dphi=360/390   # complete round = 30*13 tithis

phi=0
x0=4800
y0=4800
radmax=4500
delta=100

# outer radius for the epact disc
r1=radmax 

# initialize canvas
fig = plt.figure(figsize=(10,10))
ax = fig.add_axes([0,0,1,1])

ax.set_xlim(x0-1.1*r1,x0+1.1*r1)
ax.set_ylim(y0-1.1*r1,y0+1.1*r1)

# middle point
ax.scatter(x0,y0,marker='o',color='k')


# create epact wheel
create_wheel(ax)

if separate:
    fig.savefig('images/epact_wheel_tithis.png')

    fig2 = plt.figure(figsize=(10,10))
    ax2 = fig2.add_axes([0,0,1,1])
    ax2.set_xlim(x0-1.1*r1,x0+1.1*r1)
    ax2.set_ylim(y0-1.1*r1,y0+1.1*r1)
    # middle point
    ax2.scatter(x0,y0,marker='o',color='k')
else:
    ax2=ax

# create moons corresponding to lunar offset to the year 2000 (depak)
create_moons(ax2, depak)

if separate:
    _=ax2.axis('off')
    fig2.savefig('images/epact_wheel_moons.png')
else:
    fig.savefig('images/large_epact_wheel_'+'{0:02d}'.format(depak)+'.pdf')
